## DuckDB
DuckDB can read files from S3-backed storage through its `httpfs` extension. In managed Labs, S3-compatible endpoint and credential environment variables are usually provided by the workspace configuration.

In [ ]:
# Install duckdb
!pip install duckdb

In [ ]:
import duckdb
import os 

conn = duckdb.connect()
conn.execute("INSTALL httpfs")
conn.execute("LOAD httpfs")
conn.execute("SET s3_url_style='path'")
conn.execute(f"SET s3_endpoint='{os.environ['S3_ENDPOINT']}'")
conn.execute(f"SET s3_use_ssl = {'false' if os.environ['S3_USE_HTTPS'] == '0' else 'true'}")

To check whether DuckDB can discover credentials in the current environment, run `CALL load_aws_credentials()` after loading the `httpfs` extension.

## Iris dataset in S3 Parquet

Write the Iris dataset to an S3-backed Parquet file and read it back with DuckDB.

In [ ]:
from sklearn.datasets import load_iris

bucket = "<bucketname>"
key = "storage-examples/iris.parquet"

if not bucket or bucket == "<bucketname>":
    raise ValueError("Set bucket to your S3 bucket name.")

In [ ]:
iris = load_iris(as_frame=True).frame
s3_path = f"s3://{bucket}/{key}"

conn.register("iris", iris)
conn.execute(
    f"COPY iris TO '{s3_path}' (FORMAT PARQUET, OVERWRITE_OR_IGNORE TRUE)"
)

read_back = conn.sql(f"SELECT * FROM read_parquet('{s3_path}')").df()
read_back.head()